In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType

df = spark.table("sncf_gc.bronze.frequentation_raw")

print(f"Bronze : {df.count()} lignes")
print(f"Schema :")
df.printSchema()

In [0]:
from pyspark.sql.functions import to_date, coalesce

# 1. Déduplication
df_silver = df.dropDuplicates()
print(f"Après dédup : {df_silver.count()} lignes")

# 2. Correction des dates (deux formats : yyyy-MM-dd et dd/MM/yyyy)
df_silver = df_silver.withColumn(
    "date",
    F.to_date(
        F.coalesce(
            F.try_to_timestamp(F.col("date"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("date"), F.lit("dd/MM/yyyy"))
        )
    )
)

# 3. Typage strict
df_silver = df_silver \
    .withColumn("gare_id", F.col("gare_id").cast(IntegerType())) \
    .withColumn("heure_tranche", F.trim(F.col("heure_tranche").cast("string")).cast(IntegerType())) \
    .withColumn("nb_voyageurs", F.col("nb_voyageurs").cast(IntegerType())) \
    .withColumn("nb_non_voyageurs", F.col("nb_non_voyageurs").cast(IntegerType()))

# 4. Filtrer valeurs négatives
df_silver = df_silver.filter(
    (F.col("nb_voyageurs") >= 0) | F.col("nb_voyageurs").isNull()
)

# 5. Standardiser segment (garder seulement A, B, C)
df_silver = df_silver.filter(F.col("segment").isin(["A", "B", "C"]))

# 6. Standardiser casse
df_silver = df_silver \
    .withColumn("region", F.initcap(F.trim(F.col("region")))) \
    .withColumn("type_gare", F.initcap(F.trim(F.col("type_gare")))) \
    .withColumn("ville", F.initcap(F.trim(F.col("ville")))) \
    .withColumn("nom_gare", F.trim(F.col("nom_gare"))) \
    .withColumn("code_uic", F.trim(F.col("code_uic")))

# 7. Filtrer type_gare invalides
df_silver = df_silver.filter(F.col("type_gare").isin(["Terminus", "Passage", "Jonction"]))

print(f"Après nettoyage : {df_silver.count()} lignes")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_gare = Window.partitionBy("gare_id")

df_silver = df_silver \
    .withColumn(
        "nb_voyageurs",
        F.when(
            F.col("nb_voyageurs").isNull(),
            F.percentile_approx("nb_voyageurs", 0.5).over(window_gare)
        ).otherwise(F.col("nb_voyageurs"))
    ) \
    .withColumn(
        "nb_non_voyageurs",
        F.when(
            F.col("nb_non_voyageurs").isNull(),
            F.percentile_approx("nb_non_voyageurs", 0.5).over(window_gare)
        ).otherwise(F.col("nb_non_voyageurs"))
    ) \
    .withColumn(
        "latitude",
        F.when(
            F.col("latitude").isNull(),
            F.avg("latitude").over(window_gare)
        ).otherwise(F.col("latitude"))
    )

In [0]:
# write on Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sncf_gc.silver.frequentation_clean")

print(f"✅ silver.frequentation_clean : {spark.table('sncf_gc.silver.frequentation_clean').count()} lignes")